# CRML Fuzzing Harness Demo

Demonstrates the `FuzzHarness` against the SRI requirement model.
The harness compiles two CRML models to Modelica, builds a single
`ComparisonHarness` model that runs both implementations with the
same parametric mock, and checks for output divergence inside the
simulation (no post-hoc time-grid alignment needed).

**Sections**
1. Build & start JVM
2. Initialise harness
3. Sanity check — reference vs reference (expect all PASS)
4. Broken candidate — wrong recovery window (expect targeted FAIL)
5. Debug a single failing parameter vector

## 1  Build & start JVM

In [1]:
from pathlib import Path

ROOT = Path(".").resolve().parent  # CRML repo root
from experiments.gradle_jvm import GradleJvm

In [2]:
jvm = GradleJvm(
    project_path=ROOT,
    subproject="experiments",
    subproject_dir="submodules/experiments",
)
jvm.build()
jvm.start()

Running: /home/ubuntu/crml/vol/CRML/gradlew experiments:shadowJar  (cwd=/home/ubuntu/crml/vol/CRML)
> Task :language:generateGrammarSource UP-TO-DATE
> Task :language:compileJava UP-TO-DATE
> Task :util:generateGrammarSource NO-SOURCE
> Task :util:compileJava UP-TO-DATE
> Task :compiler:compileJava UP-TO-DATE
> Task :compiler:processResources UP-TO-DATE
> Task :compiler:classes UP-TO-DATE
> Task :compiler:jar UP-TO-DATE
> Task :experiments:compileJava NO-SOURCE
> Task :experiments:processResources UP-TO-DATE
> Task :experiments:classes UP-TO-DATE
> Task :language:processResources UP-TO-DATE
> Task :language:classes UP-TO-DATE
> Task :language:jar UP-TO-DATE
> Task :util:processResources NO-SOURCE
> Task :util:classes UP-TO-DATE
> Task :util:jar UP-TO-DATE
> Task :experiments:shadowJar UP-TO-DATE

BUILD SUCCESSFUL in 1s
11 actionable tasks: 11 up-to-date
Consider enabling configuration cache to speed up this build: https://docs.gradle.org/9.1.0/userguide/configuration_cache_enabling.htm

## 2  Initialise harness

In [3]:
from experiments.harness import CRMLCompiler, FuzzHarness
from experiments.domains import CRMLTOMODELICA_PATH, SRI2_REF_CRML, SRI_DOMAIN

compiler = CRMLCompiler()
harness  = FuzzHarness(SRI_DOMAIN, compiler, CRMLTOMODELICA_PATH)

ref_crml = SRI2_REF_CRML.read_text()
print(f"Reference CRML: {len(ref_crml)} chars, {ref_crml.count(chr(10))} lines")

Reference CRML: 11073 chars, 229 lines


## 3  Sanity check — reference vs reference

Both candidate and reference are the same source.  Every simulation run
must produce identical output signals, so the harness should report
**zero failures** across all named scenarios and random fuzz iterations.

In [4]:
result_sanity = harness.run(
    candidate_crml=ref_crml,
    reference_crml=ref_crml,
    n_iters=20,
    seed=42,
    verbose=True,
    work_dir=Path("~/crml/vol/data").expanduser(),
    keep=True,
)

print(result_sanity.summary())

[harness] Compiling candidate...
Category: null og_op : not og_op : not
Category: null og_op : and og_op : and
Category: null og_op : not og_op : not
OP DEBUG: not  Boolean b1false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
Category: null og_op : not og_op : not
OP DEBUG: not  Boolean b2false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
OP DEBUG: not  Boolean CRMLtoModelica.Functions.and4(CRMLtoModelica.Functions.not4( b2), CRMLtoModelica.Functions.not4( b1))false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
Category: null og_op : and og_op : and
Applying operator: or b2 b1 

Category: null og_op : not og_op : not
Category: null og_op : and og_op : and
OP DEBUG: not  Boolean CRMLtoModelica.Functions.and4(b2, b1)false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
Applying operator: or b2 notb1 

Category: null og_op : not og_op : not
OP DEBUG: not  Boolean b1false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
Category: null og_op : filter og_op : filter
Category: null og_

Added category increasing_int

Added category increasing_real

Added category varying1

Added category varying2

Added category increasing_int

Added category increasing_real

Added category varying1

Added category varying2



Category: null og_op : not og_op : not
Category: null og_op : and og_op : and
Category: null og_op : not og_op : not
OP DEBUG: not  Boolean b1false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
Category: null og_op : not og_op : not
OP DEBUG: not  Boolean b2false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
OP DEBUG: not  Boolean CRMLtoModelica.Functions.and4(CRMLtoModelica.Functions.not4( b2), CRMLtoModelica.Functions.not4( b1))false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
Category: null og_op : and og_op : and
Applying operator: or b2 b1 

Category: null og_op : not og_op : not
Category: null og_op : and og_op : and
OP DEBUG: not  Boolean CRMLtoModelica.Functions.and4(b2, b1)false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
Applying operator: or b2 notb1 

Category: null og_op : not og_op : not
OP DEBUG: not  Boolean b1false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
Category: null og_op : filter og_op : filter
Category: null og_op : and og_op : and
Category: nu

OMCBuildError: omc build failed — binary not produced.
stdout:
true
""
true
""
true
""
true
""
true
""
true
""
true
""
true
""
{"", ""}
"[<interactive>:110:33-110:36:writable] Notification: From here:
[<interactive>:99:1-99:34:writable] Error: Type mismatch in binding P = candidate.Req_flow1.'ensure55'.'check anytime48'.'checkover46'.P, expected subtype of CRMLtoModelica.Types.CRMLPeriod, got type CRMLtoModelica.Types.CRMLPeriods.
"

stderr:


Expected output:
```
PASS  26/26 runs matched
```
(6 named scenarios + 20 random iterations)

## 4  Broken candidate — wrong recovery window

The reference requires the temperature to return to range **within 60 s**
of an excursion.  The broken candidate uses **30 s** instead.

This only diverges when the fault duration is *between* 30 s and 60 s — e.g.
the `T_brief_fault` scenario (fault at t=30, recovery at t=80 → 50 s fault):

| model | window | check at | T in range? | R2_T |
|---|---|---|---|---|
| reference | 60 s | t = 90 | yes (recovered at 80) | **true** |
| broken    | 30 s | t = 60 | no (still faulted)    | **false** |

The `T_long_fault` scenario (no recovery) makes both models fail R2_T,
so **no mismatch** there — demonstrating that not every fault exposes the bug.

In [ ]:
# Change the recovery window from 60.0 s to 30.0 s
broken_crml = ref_crml.replace(
    "('from' (new Clock (not R1_T)) 'for' 60.0) 'check at end' T_in_range",
    "('from' (new Clock (not R1_T)) 'for' 30.0) 'check at end' T_in_range",
)

assert broken_crml != ref_crml, "String replacement had no effect — check the source"
print("Broken CRML prepared.")

In [ ]:
result_broken = harness.run(
    candidate_crml=broken_crml,
    reference_crml=ref_crml,
    n_iters=20,
    seed=42,
    verbose=True,
    work_dir=Path("~/crml/vol/data").expanduser(),
    keep=True,
)

print(result_broken.summary())

Expected output (at minimum):
```
FAIL  N/26 runs diverged
  signals: ['R2_T', 'R_T']
  params:  {'scenario': 'T_brief_fault'}
```
Random iterations with a fault lasting 30–60 s will also be flagged.

## 5  Debug — re-run a single failing vector

`run_scenario` compiles and builds once for a single parameter dict,
useful for reproducing a specific failure reported by `run()`.

In [ ]:
# Re-run the exact scenario that exposed the bug
debug_params = {
    "T_fault_start": 30.0,
    "T_recovery":    80.0,   # 50 s fault — inside reference window, outside broken window
}

result_debug = harness.run_scenario(
    candidate_crml=broken_crml,
    reference_crml=ref_crml,
    params=debug_params,
    verbose=True,
    work_dir=Path("~/crml/vol/data").expanduser(),
    keep=True,
)

print(result_debug.summary())

In [ ]:
# Confirm the long-fault scenario does NOT trigger a mismatch
# (both models fail R2_T, so they agree on the wrong answer)
result_long = harness.run_scenario(
    candidate_crml=broken_crml,
    reference_crml=ref_crml,
    params={"T_fault_start": 30.0, "T_recovery": 1e9},
    work_dir=Path("~/crml/vol/data").expanduser(),
    keep=True,
)

print("Long fault:", result_long.summary())

## 6  Semantic Analysis — mapping-driven harness run

Generated CRML models use different variable names for the same requirements
(e.g. `R1` instead of `R1_T`).  We bridge this with a per-file JSON mapping
produced by Claude (see `generated/MATCHING_PROMPT.md`).

**Workflow:**
1. For each generated `.crml`, run the prompt in `MATCHING_PROMPT.md` to produce `<name>_mapping.json`.
2. `RequirementMapping.load()` reads the JSON.
3. `mapping.apply_to_domain()` returns a `DomainSpec` with only the matched outputs,
   each `OutputSignal` carrying the generated variable name as `candidate_name`.
4. Run `FuzzHarness` normally; catch `OMCBuildError` for structurally incompatible models.

In [ ]:
from experiments.harness import FuzzHarness, OMCBuildError, RequirementMapping
from experiments.domains import CRMLTOMODELICA_PATH, SRI2_REF_CRML, SRI_DOMAIN

# Path helpers
GEN_ROOT = Path("generated")
ref_crml = SRI2_REF_CRML.read_text()

def mapping_path(crml_path: Path) -> Path:
    return crml_path.with_name(crml_path.stem + "_mapping.json")

### 6a  Single model — inspect mapping and run

In [ ]:
# Edit these two paths to analyse any generated model.
crml_file   = GEN_ROOT / "claude" / "SRI_temp_k1.crml"
mapping_file = mapping_path(crml_file)   # generated/claude/SRI_temp_k1_mapping.json

mapping = RequirementMapping.load(mapping_file)
print(mapping.report())

In [ ]:
adapted_domain  = mapping.apply_to_domain(SRI_DOMAIN)
adapted_harness = FuzzHarness(adapted_domain, compiler, CRMLTOMODELICA_PATH)
gen_crml        = crml_file.read_text()

try:
    result = adapted_harness.run(
        candidate_crml=gen_crml,
        reference_crml=ref_crml,
        n_iters=20,
        seed=42,
        verbose=True,
        work_dir=Path("~/crml/vol/data").expanduser(),
        keep=True,
    )
    print(result.summary())
except OMCBuildError as e:
    print(f"Build failed (structural incompatibility or missing externals):\n{e}")

### 6b  Batch — all models with mapping files

In [ ]:
results_table = []

for crml_file in sorted(GEN_ROOT.rglob("*.crml")):
    mp = mapping_path(crml_file)
    if not mp.exists():
        results_table.append((crml_file, "no mapping", None))
        continue

    mapping     = RequirementMapping.load(mp)
    adapted     = mapping.apply_to_domain(SRI_DOMAIN)
    gen_crml    = crml_file.read_text()
    h           = FuzzHarness(adapted, compiler, CRMLTOMODELICA_PATH)

    try:
        r = h.run(
            candidate_crml=gen_crml,
            reference_crml=ref_crml,
            n_iters=20,
            seed=42,
            verbose=False,
        )
        results_table.append((crml_file, "ok", r))
    except OMCBuildError:
        results_table.append((crml_file, "build_error", None))

print(f"{'Model':<50s} {'Status':<12s} {'Result'}")
print("-" * 80)
for path, status, r in results_table:
    label = str(path.relative_to(GEN_ROOT))
    verdict = r.summary().split("\n")[0] if r else "-"
    print(f"{label:<50s} {status:<12s} {verdict}")

## Shutdown

In [5]:
jvm.shutdown()

JVM shut down.
